# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIRˆ² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset and access metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here we will list all available record sets and their structure, referencing entities by their `@id`.

In [ ]:
# List all available record sets and their details
print("Available Record Sets:")
# Each record_set is an mlcroissant.model.RecordSet object
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, data type: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into pandas DataFrames, referencing by record set @id
record_set_ids = [rs.id for rs in metadata.record_sets]  # e.g., ['protein_record_set', ...]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
    print()

# Display first few rows of the primary record set (using the first one for demo)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"First few records for {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field (referenced by its `@id`) for demonstration.

In [ ]:
import numpy as np

# For demonstration, get field @id for a numeric field from the first record set
if main_record_set_id:
    # Find the first numeric field in the main record set
    main_rs = [rs for rs in metadata.record_sets if rs.id == main_record_set_id][0]
    numeric_fields = [f for f in main_rs.fields if f.data_type in ('Number', 'Float', 'Integer')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        numeric_field_name = numeric_fields[0].name
    else:
        numeric_field_id = None
        numeric_field_name = None

    if numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
        df = dataframes[main_record_set_id]
        # Ensure numeric type
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Filtering: Only consider values > threshold
        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} ({numeric_field_name}) > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field if available
        categorical_fields = [f for f in main_rs.fields if f.data_type in ('Text', 'String') and f.id in df.columns]
        group_field = categorical_fields[0].id if categorical_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric field found in the main record set.")
else:
    print("No main record set loaded.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and relationships between key attributes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]
    # Drop NA for plotting
    series = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()

    plt.figure(figsize=(8, 5))
    sns.histplot(series, kde=True)
    plt.title(f"Distribution of {numeric_field_id} ({numeric_field_name})")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If we have previously identified a group_field
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and analyzed the FAIRˆ² dataset of mass spectrometry protein measurements using `mlcroissant`. 

- The dataset consists of multiple record sets with detailed protein records, including abundance, modification, and sequence data from human samples.
- We inspected record set structures using their `@id`s, extracted all records dynamically, and previewed the primary table.
- Exploratory data analysis was performed on a representative numeric field, including filtering, z-score normalization, and optional grouping by a categorical attribute.
- Finally, we visualized numeric field distributions to reveal insights into the data structure.

This workflow serves as a flexible starting point for further analysis, such as biomarker selection, feature engineering, or scientific reporting on extracellular vesicle-derived proteins.